# wwgdt?
---
> 1. Ler todo o df (um ano apenas) e validar as colunas.
---
1.  Definição e Validação dos Schemas

2. Operações em Grandes Volumes de dados, incrementado gradativamente
 
2.1 - Horas

2.2 - Dias

2.3 - Semanas

2.4 - Meses

2.5 - Trimestres

2.6 - Semestres

2.7 - Anos

2.8 - Limite Máximo do Dataset

> * Operações de Consultas, com agrupamentos e Geração de Gráficos 

3. Comparação de Desempenho, envolvendo as bibliotecas utilizadas em sala

3.1 - Pandas

3.2 - PyArrow

3.3 - Polars

3.4 - Dask

4. Processamento paralelo com Dask, utilizando pelo menos 3 máquinas (Físicas) diferentes e geração de relatórios de desempenho com variações de numero de workers. Também fazer com aumento gradual, como por exemplo:

1 máquina

2 máquinas 

3 máquinas

**{ as três variar de 2 a N workers }**

In [1]:
import pandas as pd

import requests
import zipfile
import io

import fsspec
print(fsspec.__version__)

import os, warnings
os.environ['DISABLE_PANDERA_IMPORT_WARNING'] = 'True'
warnings.filterwarnings('ignore')

url = 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINASC/csv/SINASC_2020_csv.zip'

#Definindo variáveis q serão utilizadas durante a execução do código.
arquivo_local = None
url_natalidade = url
n_amostra = 500_000

fonte = arquivo_local if (arquivo_local and Path(arquivo_local).exists()) else url_natalidade
print(f'Fonte configurada: {fonte[:80]}...' if len(fonte)>80 else f'Fonte: {fonte}')


# Download do arquivo bruto - conteúdo binário
resposta_da_requisicao = requests.get(url_natalidade)
conteudo_do_arquivo = io.BytesIO(resposta_da_requisicao.content)


# "Des"compactação e leitura
with zipfile.ZipFile(conteudo_do_arquivo) as zippado:
    arquivos_internos = zippado.namelist()
    with zippado.open(arquivos_internos[0]) as f:
        df_natalidade2024 = pd.read_csv(f, sep=';', nrows=n_amostra, encoding='latin1', low_memory=False)

        
# Validação da Compressão
compressao = "ZIP" if fonte.endswith('.zip') else "Nenhuma"

# Verificação dos metadados
uso_memoria = df_natalidade2024.memory_usage(deep=True) / 1024
print(f'{"Coluna":<21}|{"Tipo":<12}|{"RAM":<5}')
print(f'{"_"*50}')
for meta in df_natalidade2024.columns:
    tipo = str(df_natalidade2024[meta].dtype)
    espaco = uso_memoria[meta]
    print(f' {meta:<20}| {tipo:<10} |{espaco:>12.1f}')
print(f'{"_"*50}')
print('')


df_natalidade2024.head()

# Separação das principais colunas para análise
lista_colunas_principais = [
    #PAIS
    'IDADEMAE',
    'IDADEPAI',

    #GESTAÇÃO
    'SEMAGESTAC', #SEMANAS DE GESTAÇÃO
    'QTDPARTNOR', #QUANTIDADE DE PARTOS NORMAIS
    'QTDPARTCES', #QUANTIDADE DE PARTOS CESÁRIOS
    'CONSPRENAT', #QUANTIDADE DE PRENATAIS FEITOS.
    'KOTELCHUCK', #INDICE QUE CLASSIFICA A QUALIDADE DO PRENATAL FEITO
    'DIFDATA', #Diferença de dias entre o nascimento e o registro.
    'GRAVIDEZ', #INDICA SE A GRAVIDEZ FOI SIMPLES, DUPLA, TRIPLA E ETC.

    #BEBÊ E NASCIMENTO
    'PESO',
    'HORANASC',
    'DTNASC',
    'SEXO'
]

# Organizando um novo dataframe com as colunas
df_final = df_natalidade2024[lista_colunas_principais].copy()
df_final.head()

#transformando em .parquet
#df_final.to_parquet('sinasc2020_reduzidissima_final.parquet', index=False)

2026.2.0
Fonte configurada: https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINASC/csv/SINASC_2020_csv....
Coluna               |Tipo        |RAM  
__________________________________________________
 ORIGEM              | int64      |      3906.2
 CODESTAB            | float64    |      3906.2
 CODMUNNASC          | int64      |      3906.2
 LOCNASC             | int64      |      3906.2
 IDADEMAE            | float64    |      3906.2
 ESTCIVMAE           | float64    |      3906.2
 ESCMAE              | float64    |      3906.2
 CODOCUPMAE          | float64    |      3906.2
 QTDFILVIVO          | float64    |      3906.2
 QTDFILMORT          | float64    |      3906.2
 CODMUNRES           | int64      |      3906.2
 GESTACAO            | float64    |      3906.2
 GRAVIDEZ            | float64    |      3906.2
 PARTO               | float64    |      3906.2
 CONSULTAS           | float64    |      3906.2
 DTNASC              | int64      |      3906.2
 HORANASC            | float

,IDADEMAE,IDADEPAI,SEMAGESTAC,QTDPARTNOR,QTDPARTCES,CONSPRENAT,KOTELCHUCK,DIFDATA,GRAVIDEZ,PESO,HORANASC,DTNASC,SEXO
0,34.0,NaN,38.0,1.0,1.0,9.0,5,42,1.0,3650.0,2035.0,3042020,2
1,18.0,NaN,NaN,1.0,0.0,NaN,9,52,1.0,1400.0,2331.0,2052020,2
2,26.0,NaN,38.0,NaN,NaN,10.0,5,8,1.0,3455.0,1020.0,6022020,1
3,29.0,NaN,39.0,2.0,0.0,6.0,4,781,1.0,3620.0,2145.0,6012020,1
4,20.0,NaN,33.0,0.0,0.0,7.0,5,762,1.0,2400.0,940.0,25012020,2


In [2]:
import pandera as pa
import pyarrow.parquet as pq

In [3]:
schema = pa.DataFrameSchema({
    'IDADEMAE': pa.Column(
        float, #dtype
        description = 'Idade da mãe do Recém Nascido.',
        nullable=True, # nullable=False: Proíbe valores vazios (NaN)
        checks=[pa.Check.gt(0), pa.Check.lt(100)],
        coerce=True
    ),
    'IDADEPAI': pa.Column(
        float, #dtype
        description = 'Idade da pai do Recém Nascido.',
        nullable = True, # nullable=False: Proíbe valores vazios (NaN)
        checks=[pa.Check.gt(0), pa.Check.lt(100)],
        coerce=True
    ),
    'SEMAGESTAC':pa.Column(
        int, #dtype
        description = 'Quantidade de semanas da gestação.',           
        nullable=True , # nullable=False: Proíbe valores vazios (NaN)
        checks=[pa.Check.gt(4), pa.Check.lt(46)],
        coerce=True
    ),
    'QTDPARTNOR':pa.Column(
        int,
        description = 'Quantidade de partos normais antes do nascimento do bebê em questão.',
        nullable = False,
        checks = pa.Check.gt(-1),
        coerce = True
    ),
    'QTDPARTCES': pa.Column(
        int,
        description = 'Quantidade de partos cesarianos antes do nascimento do bebê em questão.',
        nullable = False,
        checks = pa.Check.gt(-1),
        coerce = True
    ),
    'CONSPRENAT': pa.Column(
        int,
        description = 'Quantidade de consultas prenatais feitas.',
        nullable = False,
        checks=[pa.Check.gt(-1), pa.Check.lt(100)],
        coerce = True
    ),
    'KOTELCHUCK': pa.Column(
        int,
        description = 'Índice de Adequação do Uso de Cuidados Pré-Natais (APNCU) - Avalia a qualidade do acompanhamento médico durante a gravidez.', #Quando o pré-natal começou e quantas consultas foram feitas em relação ao esperado para a idade gestacional.'
        nullable = False,
        checks=[pa.Check.gt(0), pa.Check.lt(10)],
        coerce = True        
    ),
    'DIFDATA': pa.Column(
        int,
        description = 'Diferença entre a data de registro da criança e data de nascimento.',
        nullable = False,
        checks = pa.Check.gt(-1),
        coerce = True
    ),
    'GRAVIDEZ': pa.Column(
        int,
        description = 'Indica se a gravidez foi simples, dupla, tripla e etc.',
        checks = pa.Check.isin([1,2,3,9]),
        coerce = True
    ),
    'PESO': pa.Column(
        float,
        description = 'Peso em gramas do recém-nascido.',
        checks=[pa.Check.gt(0), pa.Check.lt(7100.0)],
        coerce = True
    ),
    'HORANASC': pa.Column(
        pa.Object,
        description='Hora do nascimento do bebê',          
        nullable=True            
    ),
    'DTNASC': pa.Column(
        "datetime64[ns]", 
        description='Data de nascimento do bebê',
    ),
    'SEXO': pa.Column(
        int,
        description = 'Sexo do recém-nascido',
        checks = pa.Check.isin([0,1,2]),
        nullable = False,
        coerce = True
    )
})

In [4]:
def Limpeza(df):
    colunas_para_limpar = df.columns.difference(['IDADEPAI', 'IDADEMAE'])
    df = df.dropna(subset=colunas_para_limpar)
    df = df[df['PESO'] > 0]
    
    df.loc[:, 'DTNASC'] = pd.to_datetime(
        df['DTNASC'].astype(int).astype(str).str.zfill(8), 
        format='%d%m%Y', 
        errors='coerce'
    )

    df.loc[:, 'HORANASC'] = pd.to_datetime(
        df['HORANASC'].astype(float).fillna(0).astype(int).astype(str).str.zfill(4), 
        format='%H%M', 
        errors='coerce'
    ).dt.time
    return df

In [5]:
arquivo = pq.ParquetFile("/home/aluno/Downloads/git/sinasc_reduzidissima_final.parquet")

print("Iniciando validação por lotes...")

try:
    for batch in arquivo.iter_batches(batch_size=n_amostra):
        df_batch = batch.to_pandas()
        df_batch = Limpeza(df_batch)
        schema.validate(df_batch)
        print(f"Lote de {len(df_batch)} linhas validado com sucesso!")
        
except pa.errors.SchemaError as e:
    print(f"❌ Erro de validação encontrado: {e}")

Iniciando validação por lotes...
Lote de 352618 linhas validado com sucesso!
Lote de 374224 linhas validado com sucesso!
Lote de 391888 linhas validado com sucesso!
Lote de 397229 linhas validado com sucesso!
Lote de 414383 linhas validado com sucesso!
Lote de 429469 linhas validado com sucesso!
Lote de 437154 linhas validado com sucesso!
Lote de 453615 linhas validado com sucesso!
Lote de 457951 linhas validado com sucesso!
Lote de 465635 linhas validado com sucesso!


In [6]:
print('Informações gerais do DF:')
print(f'{"_"*60}')
df_batch.info(memory_usage = True)
print(f'{"_"*60}')
print('Primeiras 10 linhas do DF:')
df_batch.head(10)

Informações gerais do DF:
____________________________________________________________
<class 'pandas.core.frame.DataFrame'>
Index: 465635 entries, 0 to 499999
Data columns (total 13 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   IDADEMAE    465629 non-null  float64       
 1   IDADEPAI    79360 non-null   float64       
 2   SEMAGESTAC  465635 non-null  float64       
 3   QTDPARTNOR  465635 non-null  float64       
 4   QTDPARTCES  465635 non-null  float64       
 5   CONSPRENAT  465635 non-null  float64       
 6   KOTELCHUCK  465635 non-null  int64         
 7   DIFDATA     465635 non-null  int64         
 8   GRAVIDEZ    465635 non-null  float64       
 9   PESO        465635 non-null  float64       
 10  HORANASC    465635 non-null  object        
 11  DTNASC      465635 non-null  datetime64[ns]
 12  SEXO        465635 non-null  int64         
dtypes: datetime64[ns](1), float64(8), int64(3), object(1)
memory usage:

,IDADEMAE,IDADEPAI,SEMAGESTAC,QTDPARTNOR,QTDPARTCES,CONSPRENAT,KOTELCHUCK,DIFDATA,GRAVIDEZ,PESO,HORANASC,DTNASC,SEXO
0,24.0,NaN,38.0,0.0,1.0,2.0,2,20,1.0,3120.0,08:45:00,2024-02-14,1
1,29.0,41.0,39.0,0.0,0.0,8.0,2,22,1.0,3564.0,08:50:00,2024-04-17,1
2,30.0,35.0,39.0,0.0,2.0,8.0,5,5,1.0,2816.0,10:06:00,2024-05-29,1
3,14.0,17.0,38.0,0.0,0.0,10.0,5,7,1.0,3126.0,09:00:00,2024-05-27,1
4,24.0,29.0,39.0,0.0,1.0,11.0,5,21,1.0,3622.0,08:35:00,2024-05-13,1
5,29.0,30.0,40.0,0.0,0.0,7.0,2,26,1.0,2935.0,09:15:00,2024-05-08,2
6,18.0,NaN,38.0,1.0,0.0,5.0,3,42,1.0,3365.0,01:10:00,2024-05-01,1
7,29.0,NaN,39.0,0.0,2.0,7.0,5,76,1.0,3298.0,10:05:00,2024-06-05,1
8,20.0,NaN,38.0,0.0,0.0,10.0,5,8,1.0,3240.0,05:16:00,2024-01-01,1
9,40.0,NaN,38.0,1.0,3.0,7.0,2,8,1.0,3960.0,21:23:00,2024-01-01,1


In [13]:
total_casos = df_batch[df_batch['IDADEMAE'] < 15].shape[0]
print(total_casos)

3574
